In [8]:
# ====== PASO 1: INSTALAR DEPENDENCIAS ======
!apt-get update -qq
!apt-get install -y chromium-browser chromium-chromedriver -qq

# Forzar versión estable de Chrome
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq
!apt-get install -y google-chrome-stable -qq

# Instalar librerías Python
!pip install selenium beautifulsoup4 lxml webdriver-manager -q

# ====== PASO 2: IMPORTAR LIBRERÍAS ======
import os
import time
import json
import re
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import *
from webdriver_manager.chrome import ChromeDriverManager
from google.colab import drive

# Montar Google Drive
# drive.mount('/content/drive')

# ====== CONFIGURACIÓN ======
MAX_PAGES = 5  # máximo de páginas internas a scrapear

# ====== PASO 3: CONFIGURAR DRIVER ======
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-plugins")
    options.add_argument("--disable-images")  # acelera la carga
    options.add_argument("--lang=es")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => false});")
    driver.set_page_load_timeout(25)
    return driver

# ====== PASO 4: EXTRAER TEXTO E IMÁGENES ======
def scrape_page(driver, url):
    try:
        driver.get(url)
        time.sleep(4)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, 'lxml')

        # Eliminar elementos que no aportan información
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form", "noscript", "svg"]):
            tag.decompose()

        # Extraer texto limpio
        text = soup.get_text(separator='\n')
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        clean_text = '\n'.join([phrase.strip() for line in lines for phrase in line.split("  ") if phrase.strip()])[:15000]

        # Extraer URLs de imágenes
        images = []
        for img in soup.find_all('img'):
            src = img.get('src') or img.get('data-src') or img.get('data-lazy-src') or img.get('data-original')
            if src:
                full_url = urljoin(url, src.split('?')[0])
                if full_url.startswith('http') and full_url not in images:
                    images.append(full_url)

        return {"texto": clean_text, "imagenes": images, "total_imagenes": len(images)}

    except Exception as e:
        return {"texto": f"[ERROR: {str(e)}]", "imagenes": [], "total_imagenes": 0}

# ====== PASO 5: ENCONTRAR ENLACES INTERNOS ======
def get_internal_links(driver, base_url, max_links=5):
    try:
        driver.get(base_url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'lxml')
        links = set()
        domain = urlparse(base_url).netloc

        for a in soup.find_all('a', href=True):
            href = a['href'].split('#')[0].strip()
            if not href or href.startswith(('tel:', 'mailto:', 'whatsapp:', 'javascript:', '#')):
                continue
            full_url = urljoin(base_url, href)
            if (urlparse(full_url).netloc == domain and full_url.startswith('http') and
                not any(ext in full_url.lower() for ext in ['.pdf', '.jpg', '.png', '.zip', '.doc', '.exe'])):
                links.add(full_url)

        return list(links)[:max_links]
    except:
        return []

# ====== PASO 6: SCRAPEAR SITIO COMPLETO ======
def scrape_website(url, max_pages=MAX_PAGES):
    if not url.startswith("http"):
        return {"error": "URL inválida"}

    driver = None
    try:
        driver = init_driver()
        print(f"\nScraping: {url}")

        data = {"home": scrape_page(driver, url)}
        internal_links = get_internal_links(driver, url)
        print(f"  Enlaces internos: {len(internal_links)}")

        for i, link in enumerate(internal_links[:max_pages-1]):
            print(f"  → [{i+1}] {link}")
            key = re.sub(r'\W+', '_', urlparse(link).path.strip('/')) or f"page_{i}"
            key = key[:50]
            if key in data:
                key += f"_dup{i}"
            data[key] = scrape_page(driver, link)
            time.sleep(2)

        return data

    except Exception as e:
        return {"error": f"Error general: {str(e)}"}
    finally:
        if driver:
            driver.quit()

# ====== PASO 7: FUNCIÓN PARA GUARDAR IMÁGENES ======
def download_images(images, folder_path="imagenes"):
    os.makedirs(folder_path, exist_ok=True)
    count = 0
    for i, img_url in enumerate(images):
        try:
            filename = f"img_{i+1}_{img_url.split('/')[-1].split('?')[0][:50]}"
            path = os.path.join(folder_path, filename)
            if not os.path.exists(path):
                r = requests.get(img_url, timeout=15)
                if r.status_code == 200 and 'image' in r.headers.get('Content-Type', ''):
                    with open(path, 'wb') as f:
                        f.write(r.content)
                    count += 1
        except:
            pass
    print(f"  → {count} imágenes descargadas en '{folder_path}/'")
    return count

# ====== PASO 8: DATOS A SCRAPEAR ======
websites = [
    {"nombre": "Pinteres", "website": "https://es.pinterest.com/pin/8092474327019580/"},
    {"nombre":"Libreria Anticuaria", "website":"https://www.facebook.com/libreriaanticuariarenacercultural/"}
]

# ====== PASO 9: EJECUTAR SCRAPING ======
results = []
all_text = ""

# Carpeta en Drive para guardar resultados
drive_folder = "/content/drive/MyDrive/scraping_resultados"
os.makedirs(drive_folder, exist_ok=True)

for idx, site in enumerate(websites):
    nombre = site.get("nombre", f"Negocio_{idx}")
    url = site.get("website")

    print(f"\n{'='*60}\nProcesando: {nombre}\n{'='*60}")

    if url and url.startswith("http"):
        scraped_data = scrape_website(url, MAX_PAGES)
        results.append({"nombre": nombre, "website": url, "datos_extraidos": scraped_data})

        all_text += f"\n\n{'='*60}\nNEGOCIO: {nombre}\nWEB: {url}\n{'='*60}\n"
        for page, content in scraped_data.items():
            if isinstance(content, dict):
                all_text += f"\n--- {page.upper()} ---\nIMÁGENES: {content['total_imagenes']}\n"
                for img in content['imagenes'][:5]:
                    all_text += f"  → {img}\n"
                if content['total_imagenes'] > 5:
                    all_text += f"  ... y {content['total_imagenes'] - 5} más\n"
                all_text += f"\nTEXTO:\n{content['texto'][:1500]}...\n"

        # Guardar JSON y TXT en Drive
        json_path = os.path.join(drive_folder, f"{nombre}_scraping.json")
        txt_path = os.path.join(drive_folder, f"{nombre}_scraping.txt")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(all_text)

    else:
        results.append({"nombre": nombre, "website": url, "datos_extraidos": {"error": "No tiene sitio web"}})
        all_text += f"\n\n=== {nombre} ===\nSIN SITIO WEB\n"

print("\n¡SCRAPING COMPLETADO! Los resultados se guardaron en Google Drive.")
print(f"Carpeta: {drive_folder}")



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
OK
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Procesando: Pinteres

Scraping: https://es.pinterest.com/pin/8092474327019580/
  Enlaces internos: 5
  → [1] https://es.pinterest.com/isabelsunil0168/
  → [2] https://es.pinterest.com/ideas/nighttime-ocean-painting/943900990695/
  → [3] https://es.pinterest.com/
  → [4] https://es.pinterest.com/pin/4598879027448626688/

Procesando: Libreria Anticuaria

Scraping: https://www.facebook.com/libreriaanticuariarenacercultural/
  Enlaces internos: 5
  → [1] https://www.facebook.com/libreriaanticuariarenacercultural/followers/
  → [2] https://www.facebook.com/libreriaanticuariarenacercultural/reviews
  → [3] https://ww